# Nested-WGCNA Analysis Pipeline

**Authors:** Dyugay I.A., Poslavsky A., Lukyanov D.K., Polyakov F.M., Nikitin E., Klimuk E., Dakhnovets A., Syrko D.S., Kotliar V.V., Chudakov D.M.

**Repository:** https://github.com/ilyada/NestedWGCNA

---

This notebook reproduces the key analyses from the Nested-WGCNA paper:

1. **CGM Discovery** — Coarse-grained module detection via UMAP + HDBSCAN
2. **Core Extraction** — Core decomposition to identify dense module centers
3. **CGM Annotation** — Enrichment against immune cell type signatures
4. **Bootstrap Reproducibility** — Module stability across pseudosamples (Table 1)
5. **FGM Discovery** — Fine-grained submodules via core-based normalization
6. **GO Enrichment** — Biological process annotation of FGMs (Figure 6)
7. **TF Enrichment Analysis** — Transcription factor validation of co-expression modules
8. **TF Knowledge Score** — Quantitative TF co-regulation enrichment metric
9. **Survival & Response** — Biomarker prediction analysis (Figure 10, requires ImVigor210)

**Data required:**
- `data/TCGA_BLCA.tsv` — TCGA Bladder Cancer RNA-Seq TPM *(provided)*
- `data/bostongene_signatures.txt` — BostonGene immune signatures *(provided)*
- `data/xCell_signatures.csv` — xCell cell type signatures *(provided)*
- ImVigor210 expression + clinical data — available from EGA under controlled access (NCT02108652)


## 1. Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
from tqdm import tqdm
from collections import Counter
from scipy.stats import fisher_exact
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score, v_measure_score,
    roc_auc_score, roc_curve
)

# Lifelines for survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# NestedWGCNA modules
from utils import bootstrap_filter, get_clust, get_adjacency, symmetrize
from GenFocus import genfocus

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)
sns.set_style('ticks')
%matplotlib inline

### Helper functions

In [ ]:
def preprocess(df):
    """Filter genes: remove constant expression, duplicate rows, and low-variance genes."""
    df = df.loc[:, (df != df.iloc[0]).any()]
    df = df.T.drop_duplicates().T
    df = bootstrap_filter(df)
    return df


def pc1_signature(df, genes):
    """Compute PC1 of a gene set as a per-sample score."""
    available = [g for g in genes if g in df.columns]
    if len(available) < 2:
        raise ValueError(f'Too few genes available: {available}')
    ss = StandardScaler()
    scaled = ss.fit_transform(df[available].fillna(0))
    pca = PCA(n_components=1)
    return pd.Series(pca.fit_transform(scaled)[:, 0], index=df.index)


def normalize_by_core(df, core_genes, method='spearman', top_n=50, CVR_thr=0.5):
    """
    Normalize a CGM's expression by its core gene set (ImmFocus-like normalization).

    Step 1: Compute eigengene (PC1) from core genes only.
    Step 2: Find INGS — top_n CGM genes most correlated with the eigengene.
    Step 3: Apply first normalization, compute CVR for each INGS gene.
    Step 4: Keep only INGS with CVR < CVR_thr (stable normalizers).
    Step 5: Normalize all CGM genes by final INGS.

    Parameters
    ----------
    df         : pd.DataFrame (samples x genes) — full CGM gene expression
    core_genes : list — core gene set used for eigengene computation
    top_n      : int — number of top-correlated genes to use as INGS candidates

    Returns
    -------
    (ings, normalized_df) or None
    """
    available_core = [g for g in core_genes if g in df.columns]
    if len(available_core) < 2:
        print('Too few core genes available')
        return None

    # Step 1: eigengene from core
    ss = StandardScaler()
    scaled_core = ss.fit_transform(df[available_core])
    pca = PCA(n_components=1)
    eigengene = pd.Series(pca.fit_transform(scaled_core)[:, 0], index=df.index)

    # Step 2: top_n genes most correlated with eigengene
    cor = df.corrwith(eigengene, axis=0, method=method)
    n = min(top_n, len(cor))
    ings = list(cor.nlargest(n).index)
    print(f'INGS candidates (top {n} by {method} correlation): {len(ings)}')

    # Step 3: first normalization + CVR computation
    fings = df[ings].mean(axis=1)
    df_norm = df.copy()
    non_ings = [g for g in df.columns if g not in ings]
    df_norm[non_ings] = df_norm[non_ings].div(fings, axis=0)
    for j in ings:
        other = [g for g in ings if g != j]
        df_norm[j] = df[j].div(df[other].mean(axis=1))

    cvr = {}
    for gene in ings:
        orig_mean = df[gene].mean()
        norm_mean = df_norm[gene].mean()
        if orig_mean == 0 or norm_mean == 0:
            continue
        orig_cv = df[gene].std() / orig_mean
        norm_cv = df_norm[gene].std() / norm_mean
        if orig_cv != 0:
            cvr[gene] = norm_cv / orig_cv

    # Step 4: CVR filter
    ings_final = [g for g in ings if cvr.get(g, np.inf) < CVR_thr]
    print(f'INGS after CVR filter (<{CVR_thr}): {len(ings_final)}')
    if len(ings_final) == 0:
        return None

    # Step 5: final normalization
    fings_final = df[ings_final].mean(axis=1)
    df_final = df.copy()
    non_ings_final = [g for g in df.columns if g not in ings_final]
    df_final[non_ings_final] = df_final[non_ings_final].div(fings_final, axis=0)
    for j in ings_final:
        other = [g for g in ings_final if g != j]
        if len(other) > 0:
            df_final[j] = df[j].div(df[other].mean(axis=1))

    return ings_final, df_final

## 2. Data Loading and Preprocessing

We use TCGA Bladder Cancer (BLCA) RNA-Seq data (TPM, 406 cases) as the primary dataset.
Genes are filtered using the bootstrap filter: genes with misclassification error < 1/e
carry insufficient information for bootstrap-based downstream analysis.

In [ ]:
data = pd.read_csv('data/TCGA_BLCA.tsv', sep='\t', index_col=0)
print(f'Raw data: {data.shape[0]} samples x {data.shape[1]} genes')

clear_data = preprocess(data)
print(f'After preprocessing: {clear_data.shape[0]} samples x {clear_data.shape[1]} genes')

## 3. Coarse-Grained Module (CGM) Discovery

**Key improvements over standard WGCNA:**

| Step | Standard WGCNA | Nested-WGCNA |
|------|---------------|----------------|
| Adjacency | `|r|^beta` (arbitrary beta) | `r^2` (coefficient of determination) |
| Dissimilarity | Szymkiewicz-Simpson (not a metric) | `sqrt(1 - r^2)` (proper metric) |
| Clustering | Hierarchical + Dynamic Tree Cut | UMAP + HDBSCAN |

The only user-facing parameter is `min_clust_size` — the minimum number of genes per module.
UMAP components are set dynamically to `log2(n_samples) + 1`.

In [ ]:
import time
set_seed(42)

# min_clust_size: minimum genes per module — raise to get fewer, larger modules
# n_components:   UMAP dimensions — set to log2(n_samples)+1 by default
# Typical runtime on ~5000 genes / 400 samples: 3-8 minutes
t0 = time.time()
clusters, core_clusters = get_clust(clear_data, min_clust_size=100, n_components=6)
print(f'Total time: {(time.time() - t0) / 60:.1f} min')

modules = pd.DataFrame({'module': clusters}, index=clear_data.columns)
cores = pd.DataFrame({'module': core_clusters}, index=clear_data.columns)

n_modules = clusters.max() + 1
n_assigned = (clusters != -1).sum()
n_core_assigned = (core_clusters != -1).sum()

print(f'Total CGMs discovered:   {n_modules}')
print(f'Genes in CGMs (HDBSCAN): {n_assigned} / {len(clusters)}')
print(f'Genes in CGM cores:      {n_core_assigned} / {len(clusters)}')

In [ ]:
# Module size summary table
module_ids = sorted([i for i in Counter(clusters).keys() if i != -1])

module_sizes = pd.DataFrame({
    'CGM': module_ids,
    'HDBSCAN size': [Counter(clusters)[i] for i in module_ids],
    'Core size':    [Counter(core_clusters)[i] for i in module_ids],
})
module_sizes['Core / HDBSCAN'] = (
    module_sizes['Core size'] / module_sizes['HDBSCAN size']
).round(2)

display(module_sizes)

# Visualize
fig, ax = plt.subplots(figsize=(12, 4), dpi=100)
x = np.arange(len(module_ids))
ax.bar(x - 0.2, module_sizes['HDBSCAN size'], width=0.4, label='HDBSCAN', alpha=0.8)
ax.bar(x + 0.2, module_sizes['Core size'],    width=0.4, label='Core',    alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'CGM-{i}' for i in module_ids], rotation=45, ha='right')
ax.set_ylabel('Number of genes')
ax.set_title('CGM and Core sizes (TCGA BLCA)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. CGM Core Annotation

CGM cores are annotated via hypergeometric enrichment test against xCell and BostonGene
immune cell type signatures. The background is the full set of analyzed genes.

In [ ]:
# Load reference signatures
signatures = {}

with open('data/bostongene_signatures.txt') as f:
    for line in f.readlines():
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            signatures[parts[0] + '_bostongen'] = parts[2:]

with open('data/xCell_signatures.csv') as f:
    for line in f.readlines()[1:]:
        parts = line.strip().split(',')
        if len(parts) >= 3:
            signatures[parts[0]] = list(filter(None, parts[2:]))

print(f'Loaded {len(signatures)} reference signatures')

In [ ]:
cgm_annotations = {}   # top hit term per CGM
cgm_enr_results  = {}   # full significant results DataFrame per CGM
background = list(modules.index)

print('=== CGM Core Annotation ===')
print(f'Testing {len([i for i in cores["module"].unique() if i != -1])} modules against {len(signatures)} signatures...\n')
for i in sorted(cores['module'].unique()):
    if i == -1:
        continue
    gene_list = list(cores[cores['module'] == i].index)
    if len(gene_list) == 0:
        cgm_annotations[i] = 'no core genes'
        print(f'  CGM-{i:2d}: no core genes after erosion — skipping')
        continue
    try:
        enr = gp.enrich(
            gene_list=gene_list,
            gene_sets=signatures,
            background=background,
            outdir=None,
            verbose=False
        )
        sig = enr.results[enr.results['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
        cgm_enr_results[i] = sig
        if len(sig) > 0:
            cgm_annotations[i] = sig.iloc[0]['Term']
            print(f'  CGM-{i:2d} ({len(gene_list):4d} core genes): {len(sig)} significant hits')
            for _, row in sig.iterrows():
                print(f'      {row["Term"]:55s}  adj.p={row["Adjusted P-value"]:.2e}  OR={row["Odds Ratio"]:.1f}')
        else:
            cgm_annotations[i] = 'unannotated'
            print(f'  CGM-{i:2d} ({len(gene_list):4d} core genes): no significant enrichment')
    except Exception as e:
        cgm_annotations[i] = 'error'
        print(f'  CGM-{i:2d}: error — {e}')

## 5. CGM Core Reproducibility (Bootstrap Analysis)

Module stability is assessed by generating bootstrap pseudosamples (~63.2% unique
samples each) and comparing clustering solutions pairwise using Adjusted Rand Index
(ARI), Adjusted Mutual Information (AMI), and V-measure.

**Table 1 from the paper** (TCGA BLCA, 10 bootstrap samples):

| Method | mARI | CV(ARI) | mAMI | CV(AMI) |
|--------|------|---------|------|---------|
| WGCNA      | 0.21 | 23% | 0.42 | 9% |
| CGM cores  | 0.42 |  9% | 0.47 | 6% |

> **Note:** Running the full bootstrap takes ~30–60 minutes depending on hardware.
> Set `n_samples=3` for a quick demonstration, or `n_samples=10` to reproduce Table 1.

In [ ]:
def bootstrap_reproducibility(data, n_samples=5, min_clust_size=100, n_components=6, seed=42):
    """
    Assess CGM core reproducibility via bootstrap resampling.

    For each of n_samples bootstrap replicates:
      - Draw samples with replacement (~63.2% unique)
      - Run full Nested-WGCNA pipeline
      - Record core cluster assignments

    Returns mean and CV of ARI, AMI, V-measure across all pairwise comparisons.
    """
    np.random.seed(seed)
    all_cores = []

    print(f'Generating {n_samples} bootstrap pseudosamples...')
    for i in tqdm(range(n_samples)):
        idx = np.random.choice(len(data), size=len(data), replace=True)
        pseudo = data.iloc[np.unique(idx)]  # ~63.2% unique samples
        _, core_clust = get_clust(pseudo, min_clust_size=min_clust_size,
                                  n_components=n_components)
        all_cores.append(pd.Series(core_clust, index=pseudo.columns))

    # Pairwise comparison on shared genes
    aris, amis, vmeas = [], [], []
    gene_set = set(data.columns)

    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            shared = list(gene_set & set(all_cores[i].index) & set(all_cores[j].index))
            a = all_cores[i][shared].values
            b = all_cores[j][shared].values
            aris.append(adjusted_rand_score(a, b))
            amis.append(adjusted_mutual_info_score(a, b))
            vmeas.append(v_measure_score(a, b))

    cv = lambda x: np.std(x) / np.mean(x) * 100 if np.mean(x) != 0 else np.nan
    return pd.DataFrame([{
        'mARI':   round(np.mean(aris),  3),
        'CV(ARI)': f'{cv(aris):.0f}%',
        'mAMI':   round(np.mean(amis),  3),
        'CV(AMI)': f'{cv(amis):.0f}%',
        'mV':     round(np.mean(vmeas), 3),
        'CV(V)':   f'{cv(vmeas):.0f}%',
    }])


# Uncomment to run (slow):
# bootstrap_results = bootstrap_reproducibility(clear_data, n_samples=5)
# display(bootstrap_results)

# Reference values from paper:
display(pd.DataFrame([
    {'Method': 'WGCNA',     'mARI': 0.21, 'CV(ARI)': '23%', 'mAMI': 0.42, 'CV(AMI)': '9%'},
    {'Method': 'CGM cores', 'mARI': 0.42, 'CV(ARI)':  '9%', 'mAMI': 0.47, 'CV(AMI)': '6%'},
]).set_index('Method'))

## 6. Fine-Grained Module (FGM) Discovery

For each CGM, genes are normalized by the CGM core (removing the dominant expression
component), then co-expression network analysis is repeated. The resulting FGMs
represent gene sets that remain correlated *independently* of the core process —
corresponding to cellular subprocesses or cell subsets within the CGM.

> *Social analogy (from the paper):* To find genomic bioinformaticians (FGM) on a
> campus of 10,000 scientists (CGM), ignore shared traits like lab coats and late hours,
> and focus on unique habits — like staring at a Jupyter notebook.

In [ ]:
# Select the immune CGM — CGM with the most immune-related significant hits
IMMUNE_KEYWORDS = ['cd4', 'cd8', 't cell', 't-cell', 'nk cell', 'nk_cell',
                    'b cell', 'b-cell', 'lymphocyte', 'lymph', 'immune',
                    'pbmc', 'monocyte', 'macrophage', 'dendritic', 'tregs',
                    'basophil', 'neutrophil', 'mast cell']

immune_scores = {}
for idx, sig_df in cgm_enr_results.items():
    if sig_df is not None and len(sig_df) > 0:
        n_immune = sum(
            any(kw in term.lower() for kw in IMMUNE_KEYWORDS)
            for term in sig_df['Term']
        )
        immune_scores[idx] = n_immune

if immune_scores:
    immune_cgm_idx = max(immune_scores, key=immune_scores.get)
    print(f'Immune CGM: CGM-{immune_cgm_idx} '
          f'({immune_scores[immune_cgm_idx]} immune hits, '
          f'top: {cgm_annotations[immune_cgm_idx]})')
else:
    # Fallback: largest annotated module
    immune_cgm_idx = Counter(
        {k: v for k, v in Counter(clusters).items() if k != -1}
    ).most_common(1)[0][0]
    print(f'No immune annotation found; using largest CGM-{immune_cgm_idx}')

cgm_genes  = modules[modules['module'] == immune_cgm_idx].index.tolist()
core_genes = cores[cores['module'] == immune_cgm_idx].index.tolist()
sub_df = clear_data[cgm_genes]

print(f'  CGM genes:  {len(cgm_genes)}')
print(f'  Core genes: {len(core_genes)}')

# Normalize by core
# corr_thr: Spearman correlation threshold for INGS selection (lower = more INGS)
# CVR_thr:  max allowed ratio of CV after/before normalization (lower = stricter)
print('Running core-based normalization...')
t0 = time.time()
result = normalize_by_core(sub_df, core_genes, top_n=50, CVR_thr=0.5)
print(f'Normalization time: {time.time() - t0:.1f} s')

if result is not None:
    ings, normalized_df = result
    print(f'\nFinal INGS: {ings}')
else:
    print('Normalization failed. Try adjusting corr_thr or CVR_thr.')

In [ ]:
# Discover FGMs on the normalized expression (INGS genes excluded)
# min_clust_size is smaller here (10) because FGMs are compact by design
fgm_input = normalized_df.dropna(axis=0).drop(columns=ings, errors='ignore')
print(f'FGM input: {fgm_input.shape[1]} genes x {fgm_input.shape[0]} samples')

set_seed(42)
t0 = time.time()
fgm_clusters, fgm_core_clusters = get_clust(
    fgm_input, min_clust_size=10, n_components=6
)
print(f'Total time: {(time.time() - t0) / 60:.1f} min')

fgm_modules = pd.DataFrame({'module': fgm_clusters}, index=fgm_input.columns)

n_fgms = fgm_clusters.max() + 1
print(f'FGMs discovered: {n_fgms}')
print(f'Genes assigned:  {(fgm_clusters != -1).sum()} / {len(fgm_clusters)}')

# Size table
fgm_ids = sorted([i for i in Counter(fgm_clusters).keys() if i != -1])
fgm_sizes = pd.DataFrame({
    'FGM': fgm_ids,
    'Size': [Counter(fgm_clusters)[i] for i in fgm_ids]
})
display(fgm_sizes)

## 7. FGM Enrichment Analysis

Each FGM is tested against two reference sets:

1. **GO Biological Process** — confirms that submodules correspond to coherent biological functions (Figure 6 in the paper)
2. **Cell type signatures** (BostonGene / xCell) — same approach as CGM annotation; identifies immune cell subtype identity of each FGM

In [ ]:
def enrichment_analysis(gene_list, background, gene_sets, alpha=0.05, top_n=5):
    """Run overrepresentation analysis and return significant terms."""
    enr = gp.enrich(
        gene_list=gene_list,
        gene_sets=gene_sets,
        background=background,
        outdir=None,
        verbose=False
    )
    sig = enr.results[enr.results['Adjusted P-value'] < alpha]
    return sig.sort_values('Adjusted P-value').head(top_n)[
        ['Term', 'Overlap', 'Adjusted P-value', 'Odds Ratio']
    ]

In [ ]:
# Download GO Biological Process gene sets (requires internet)
print('Downloading GO_Biological_Process_2021 from Enrichr...')
go_bp = gp.get_library(name='GO_Biological_Process_2021', organism='Human')
print(f'Loaded {len(go_bp)} GO terms.')

print('\n=== FGM GO Enrichment ===')
print(f'Testing {len(fgm_ids)} FGMs against GO Biological Process...\n')
fgm_background = list(fgm_modules.index)
fgm_go_annotations = {}

for i in fgm_ids:
    gene_list = list(fgm_modules[fgm_modules['module'] == i].index)
    try:
        results = enrichment_analysis(gene_list, fgm_background, go_bp)
        if len(results) > 0:
            top_term = results.iloc[0]['Term']
            fgm_go_annotations[i] = top_term
            print(f'\n  FGM-{i} ({len(gene_list)} genes): {top_term}')
            display(results)
        else:
            fgm_go_annotations[i] = 'unannotated'
            print(f'  FGM-{i}: no significant GO terms')
    except Exception as e:
        fgm_go_annotations[i] = 'error'
        print(f'  FGM-{i}: error — {e}')

In [ ]:
# Annotate FGMs against cell type signatures (same as CGM annotation)
print('=== FGM Cell Type Annotation ===')
print(f'Testing {len(fgm_ids)} FGMs against {len(signatures)} signatures...\n')
fgm_sig_annotations = {}

for i in fgm_ids:
    gene_list = list(fgm_modules[fgm_modules['module'] == i].index)
    if len(gene_list) == 0:
        fgm_sig_annotations[i] = 'no genes'
        print(f'  FGM-{i}: no genes — skipping')
        continue
    try:
        enr = gp.enrich(
            gene_list=gene_list,
            gene_sets=signatures,
            background=fgm_background,
            outdir=None,
            verbose=False
        )
        sig = enr.results[enr.results['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
        if len(sig) > 0:
            fgm_sig_annotations[i] = sig.iloc[0]['Term']
            print(f'  FGM-{i} ({len(gene_list):3d} genes): {len(sig)} significant hits')
            for _, row in sig.iterrows():
                print(f'      {row["Term"]:55s}  adj.p={row["Adjusted P-value"]:.2e}  OR={row["Odds Ratio"]:.1f}')
        else:
            fgm_sig_annotations[i] = 'unannotated'
            print(f'  FGM-{i} ({len(gene_list):3d} genes): no significant enrichment')
    except Exception as e:
        fgm_sig_annotations[i] = 'error'
        print(f'  FGM-{i}: error — {e}')

## 8. Transcription Factor Enrichment Analysis

### Motivation

Co-expression modules reflect **mRNA-level** correlations. Transcription factors (TFs)
directly control transcription, so genes in a meaningful co-expression module should
share common TF regulators. This is a more mechanistically appropriate validation than
protein-level pathway databases (e.g., Reactome), where the gene–protein expression
correlation is generally weak.

### Data used

> **Note:** The enrichment analysis below is run on **precomputed module assignments**
> from the IMvigor210 bladder cancer dataset (Rosenberg et al., 2016), which is available
> only under controlled access via the European Genome-Phenome Archive. The precomputed
> cluster files are provided in the `data/` directory.
>
> To run this analysis on **your own data**, replace the `DATASETS` list entries with
> your own `pd.Series(gene -> module_id)` objects (noise genes should have label `-1`).

### TF library used

| Library | Description |
|---------|-------------|
| **ChEA_2022** | TF→target from ChIP-seq / ChIP-chip experiments (gold standard) |

All enrichment is performed as overrepresentation analysis (ORA) with
Benjamini–Hochberg correction, using the full set of analyzed genes as background.

In [ ]:
def load_tf_libraries(library_names=None):
    """Download TF libraries once and return as a dict {name -> gene_set_dict}."""
    if library_names is None:
        library_names = [
            'ChEA_2022',
            'ENCODE_TF_ChIP-seq_2015',
            'TF_Perturbations_Followed_by_Expression',
        ]
    loaded = {}
    for name in library_names:
        print(f'  Downloading {name}...')
        lib = gp.get_library(name=name, organism='Human')
        # ChEA and ENCODE libraries mix human and mouse experiments;
        # keep only gene sets whose name does not contain 'MOUSE'
        lib = {k: v for k, v in lib.items() if 'MOUSE' not in k.upper()}
        loaded[name] = lib
        print(f'  {name}: {len(loaded[name])} gene sets (human only)')
    return loaded


def run_tf_enrichment(gene_list, background, tf_libraries, alpha=0.05, top_n=5):
    """
    Run TF overrepresentation analysis using pre-loaded TF libraries.

    Parameters
    ----------
    gene_list    : list of gene symbols in the module
    background   : list of all analyzed genes (universe)
    tf_libraries : dict {library_name -> gene_set_dict} from load_tf_libraries()
    alpha        : FDR threshold

    Returns
    -------
    dict: {library_name -> DataFrame of significant TF terms}
    """
    results = {}
    for lib_name, tf_lib in tf_libraries.items():
        try:
            enr = gp.enrich(
                gene_list=gene_list,
                gene_sets=tf_lib,
                background=background,
                outdir=None,
                verbose=False
            )
            sig = enr.results[enr.results['Adjusted P-value'] < alpha]
            results[lib_name] = sig.sort_values('Adjusted P-value').head(top_n)[
                ['Term', 'Overlap', 'Adjusted P-value', 'Odds Ratio']
            ]
        except Exception as e:
            results[lib_name] = pd.DataFrame({'error': [str(e)]})
    return results

In [ ]:
# Load precomputed ImVigor210 module assignments
# WGCNA: dynamicMods=0 is grey (unassigned) → remap to -1
wgcna_raw = pd.read_csv('data/ImVigor_clusters.tsv', index_col=0)
wgcna_series = wgcna_raw.set_index('colnames.imvigor.')['dynamicMods'].replace(0, -1)

nw_modules = pd.read_csv('data/ImVigor210_WGCNA_modules.tsv',
                         sep='\t', index_col=0)['module']
nw_cores = pd.read_csv('data/ImVigor210_WGCNA_cores.tsv',
                       sep='\t', index_col=0)['module']

# Parse precomputed ImVigor210 FGMs (immune CGM)
import re
fgm_path = 'data/Supplementary Data 4 FGM ImVigor_immune.txt'
with open(fgm_path, encoding='utf-8') as f:
    fgm_text = f.read()
fgm_module_genes = {}
for block in re.split(r'\n(?=Module \d+)', fgm_text.strip()):
    m = re.match(r'Module (\d+).*?genes:\n(.+)', block)
    if m:
        mod_id = int(m.group(1))
        fgm_module_genes[mod_id] = [g.strip() for g in m.group(2).split(',')]
fgm_records = [(g, mid) for mid, genes in fgm_module_genes.items() for g in genes]
fgm_series = pd.Series(dict(fgm_records))
print(f'Loaded {len(fgm_module_genes)} precomputed FGMs, {len(fgm_series)} genes')

# Download TF libraries once — reused for all datasets and modules
print('Loading TF libraries (once)...')
tf_libs = load_tf_libraries(['ChEA_2022'])

# Background = all genes present in each file (including unassigned),
# matching the full analyzed gene universe
DATASETS = [
    ('WGCNA',                              wgcna_series, list(wgcna_series.index)),
    ('NestedWGCNA CGM modules',            nw_modules,   list(nw_modules.index)),
    ('NestedWGCNA CGM cores',              nw_cores,     list(nw_modules.index)),
    ('NestedWGCNA FGMs (immune ImVigor210)', fgm_series, list(fgm_series.index)),
]

tf_enrichment_results = {}

for label, module_series, background in DATASETS:
    module_ids = sorted(m for m in module_series.unique() if m != -1)
    print(f'=== TF Enrichment: {label} ({len(module_ids)} modules) ===\n')
    results = {}
    for i in module_ids:
        gene_list = list(module_series[module_series == i].index)
        if len(gene_list) == 0:
            continue
        tf_res = run_tf_enrichment(gene_list, background=background,
                                   tf_libraries=tf_libs)
        results[i] = tf_res
        chea = tf_res.get('ChEA_2022', pd.DataFrame())
        print(f'  Module-{i} ({len(gene_list)} genes):')
        if len(chea) > 0 and 'Term' in chea.columns:
            for _, row in chea.iterrows():
                print(f'    {row["Term"]:45s}  adj.p={row["Adjusted P-value"]:.2e}  OR={row["Odds Ratio"]:.1f}')
        else:
            print('    No significant TFs found')
    tf_enrichment_results[label] = results
    print()

## 9. TF Knowledge Score

A single summary statistic that quantifies whether gene pairs within the same module
are enriched for sharing a common TF regulator — a global measure of module TF coherence.

**Definitions:**
- **Co-expression Pair (CP):** Two genes assigned to the same module
- **TF Knowledge Pair (KP):** Two genes sharing ≥1 TF in ChEA_2022

**Score formula** (Abbassi-Daloii et al., 2020):
$$\text{Score} = \frac{|CP \cap KP| \times |\overline{CP} \cap \overline{KP}|}{|\overline{CP} \cap KP| \times |CP \cap \overline{KP}|}$$

A score > 1 means co-expressed pairs are more likely to share a TF regulator than
expected by chance. Statistical significance via Fisher's exact test.

> **Note:** The scores below are computed on **precomputed IMvigor210 cluster assignments**
> (loaded in Section 8). The `compute_tf_knowledge_score` function accepts any
> `pd.Series(gene -> module_id)` — pass your own module assignments to reproduce
> this analysis on a different dataset.

In [ ]:
def compute_tf_knowledge_score(module_series, tf_gene_sets, min_annotated=50):
    """
    Compute TF-based knowledge score for co-expression modules.

    Parameters
    ----------
    module_series : pd.Series  (gene -> module_id), noise genes have label -1
    tf_gene_sets  : dict       {TF_name -> [target_gene, ...]}
    min_annotated : int        minimum TF-annotated genes required

    Returns
    -------
    dict with contingency table, knowledge score, and p-value
    """
    # Build gene -> TF set mapping (invert TF -> genes)
    gene_to_tfs = {}
    for tf, targets in tf_gene_sets.items():
        tf_clean = tf.split('_')[0].upper()  # normalize TF name
        for gene in targets:
            gene_to_tfs.setdefault(gene, set()).add(tf_clean)

    # Filter to module genes (non-noise) with TF annotation
    module_genes = module_series[module_series != -1]
    annotated_genes = [g for g in module_genes.index if g in gene_to_tfs]
    if len(annotated_genes) < min_annotated:
        print(f'Only {len(annotated_genes)} TF-annotated genes — insufficient.')
        return None

    print(f'TF-annotated genes: {len(annotated_genes)} / {len(module_genes)}')

    labels = module_series[annotated_genes].values
    n = len(annotated_genes)

    # Build binary gene x TF matrix for fast pair computation
    all_tfs = sorted(set(tf for tfs in gene_to_tfs.values() for tf in tfs))
    tf_to_idx = {tf: i for i, tf in enumerate(all_tfs)}
    T = len(all_tfs)

    print(f'Building gene x TF matrix ({n} x {T})...')
    M = np.zeros((n, T), dtype=np.float32)
    for i, gene in enumerate(annotated_genes):
        for tf in gene_to_tfs[gene]:
            if tf in tf_to_idx:
                M[i, tf_to_idx[tf]] = 1.0

    # KP matrix: gene pair shares ≥1 TF  <=>  dot product > 0
    print('Computing pairwise TF overlap (matrix multiply)...')
    K = M @ M.T           # shape (n, n)
    KP = (K > 0)           # True if pair shares at least one TF
    np.fill_diagonal(KP, False)

    # CP matrix: same module
    CP = (labels[:, None] == labels[None, :]) & (labels[:, None] != -1)
    np.fill_diagonal(CP, False)

    # Count upper-triangle pairs only (avoid double counting)
    upper = np.triu(np.ones((n, n), dtype=bool), k=1)
    cp_kp   = int((CP &  KP & upper).sum())
    cp_nkp  = int((CP & ~KP & upper).sum())
    ncp_kp  = int((~CP &  KP & upper).sum())
    ncp_nkp = int((~CP & ~KP & upper).sum())

    contingency = [[cp_kp, cp_nkp], [ncp_kp, ncp_nkp]]
    _, p_val = fisher_exact(contingency, alternative='greater')

    score = (cp_kp * ncp_nkp) / (ncp_kp * cp_nkp) if (ncp_kp * cp_nkp) > 0 else np.nan

    return {
        'CP∩KP':      cp_kp,
        'CP∩¬KP':     cp_nkp,
        '¬CP∩KP':     ncp_kp,
        '¬CP∩¬KP':    ncp_nkp,
        'TF Knowledge Score': round(score, 3) if not np.isnan(score) else None,
        'p-value':    p_val,
    }

In [ ]:
# -----------------------------------------------------------------------
# PRECOMPUTED DATA: wgcna_series, nw_modules, nw_cores, fgm_series were
# loaded in Section 8 from the data/ precomputed files (IMvigor210).
# To use your own modules, replace these variables with your own
# pd.Series(gene -> module_id), then add entries to the `rows` list below.
# -----------------------------------------------------------------------

# Load ChEA_2022 (requires internet)
print('Downloading ChEA_2022 from Enrichr...')
chea_lib = gp.get_library(name='ChEA_2022', organism='Human')
chea_lib = {k: v for k, v in chea_lib.items() if 'MOUSE' not in k.upper()}
print(f'ChEA_2022: {len(chea_lib)} human TF gene sets loaded')

# --- Compute TF knowledge scores ---
rows = []

print('\n--- WGCNA TF Knowledge Score ---')
s = compute_tf_knowledge_score(wgcna_series, chea_lib)
if s:
    rows.append({'Method': 'WGCNA', **s})

print('\n--- NestedWGCNA CGM (HDBSCAN) TF Knowledge Score ---')
s = compute_tf_knowledge_score(nw_modules, chea_lib)
if s:
    rows.append({'Method': 'NestedWGCNA CGM modules', **s})

print('\n--- NestedWGCNA CGM Cores TF Knowledge Score ---')
s = compute_tf_knowledge_score(nw_cores, chea_lib)
if s:
    rows.append({'Method': 'NestedWGCNA CGM cores', **s})

print('\n--- NestedWGCNA FGMs (immune CGM, ImVigor210) TF Knowledge Score ---')
s = compute_tf_knowledge_score(fgm_series, chea_lib)
if s:
    rows.append({'Method': 'NestedWGCNA FGMs (immune)', **s})

# --- Summary ---
print('\n=== TF Knowledge Score Comparison (ImVigor210) ===')
display(pd.DataFrame(rows).set_index('Method'))

## 10. Survival and Response Analysis

> **Data access:** This section requires the ImVigor210 clinical trial data
> (NCT02108652), available from the European Genome-Phenome Archive (EGA)
> under controlled access. The expression matrix and clinical metadata
> (overall survival, censoring, binary response) must be obtained separately.

This section reproduces **Figure 10** from the paper: Kaplan-Meier survival
curves and response boxplots for FGM4 (cytotoxic) and normalized FGM/core ratios
in bladder cancer patients treated with atezolizumab (anti-PD-L1).

In [ ]:
import os

# ---- DATA LOADING (requires EGA access) ----
# imvigor_expr = pd.read_csv('data/ImVigor210_TPM.tsv', sep='\t', index_col=0)
# metadata = pd.read_csv('data/ImVigor210_metadata.tsv', sep='\t', index_col=0)
# Columns needed in metadata: 'os' (float), 'censOS' (0/1), 'binary_response' (0/1)

IMVIGOR_AVAILABLE = os.path.exists('data/ImVigor210_TPM.tsv')
print(f'ImVigor210 data available: {IMVIGOR_AVAILABLE}')
print('To run this section, place ImVigor210_TPM.tsv and ImVigor210_metadata.tsv in data/')

In [ ]:
def survival_analysis(data, metadata, signature_genes, label, ax_km=None, ax_box=None):
    """
    Compute PC1 signature score and plot Kaplan-Meier + response boxplot.

    Parameters
    ----------
    data           : pd.DataFrame (samples x genes) — expression
    metadata       : pd.DataFrame — must contain 'os', 'censOS', 'binary_response'
    signature_genes: list of gene symbols
    label          : str — name for plot titles
    """
    # Compute signature score (PC1)
    available = [g for g in signature_genes if g in data.columns]
    score = pc1_signature(data, available)

    df = metadata[['os', 'censOS', 'binary_response']].copy()
    df['score'] = score
    df = df.dropna()

    high = df['score'] >= df['score'].median()

    # Kaplan-Meier
    if ax_km is None:
        _, ax_km = plt.subplots(figsize=(5, 4), dpi=100)

    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[high, 'os'], df.loc[high, 'censOS'], label='High')
    kmf.plot_survival_function(ax=ax_km, ci_show=False, color='#d73027')
    kmf.fit(df.loc[~high, 'os'], df.loc[~high, 'censOS'], label='Low')
    kmf.plot_survival_function(ax=ax_km, ci_show=False, color='#1a9850')

    p = logrank_test(df.loc[high, 'os'], df.loc[~high, 'os'],
                     df.loc[high, 'censOS'], df.loc[~high, 'censOS']).p_value
    ax_km.set_title(f'{label}\nlog-rank p = {p:.3f}')
    ax_km.set_xlabel('Overall survival (days)')
    ax_km.set_ylabel('Survival probability')

    # Response boxplot
    resp_df = df[df['binary_response'].notna()].copy()
    resp_df['Response'] = resp_df['binary_response'].map({1: 'Response', 0: 'Non-Response'})

    if ax_box is None:
        _, ax_box = plt.subplots(figsize=(3, 4), dpi=100)

    sns.boxplot(
        data=resp_df, x='Response', y='score',
        order=['Response', 'Non-Response'],
        palette={'Response': '#1a9850', 'Non-Response': '#d73027'},
        ax=ax_box
    )
    auc = roc_auc_score(resp_df['binary_response'],
                        resp_df['score'].where(resp_df['binary_response'] == 1,
                                               -resp_df['score']))
    ax_box.set_title(f'{label}\nAUC = {auc:.2f}')
    ax_box.set_xlabel('')
    ax_box.set_ylabel('Signature score')

    plt.tight_layout()

    return {'p_logrank': p, 'AUC': auc}

In [ ]:
# FGM4 — cytotoxic signature (from paper)
# Includes: IFNG, SLFN12L, TNFRSF9, CD2, PYHIN1, ITGAL, EOMES, CCR5, CD3G,
#           GZMK, TTC24, PTPN22, NKG7, PDCD1, TBX21, GZMB, ABCD2, GZMA, CD8A

FGM4_CYTOTOXIC = [
    'IFNG', 'SLFN12L', 'TNFRSF9', 'CD2', 'PYHIN1', 'ITGAL', 'EOMES', 'CCR5',
    'CD3G', 'GZMK', 'TTC24', 'PTPN22', 'NKG7', 'PDCD1', 'TBX21', 'GZMB',
    'ABCD2', 'GZMA', 'CD8A'
]

if IMVIGOR_AVAILABLE:
    imvigor_expr = pd.read_csv('data/ImVigor210_TPM.tsv', sep='\t', index_col=0)
    metadata = pd.read_csv('data/ImVigor210_metadata.tsv', sep='\t', index_col=0)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=100)

    # FGM4 unnormalized
    res1 = survival_analysis(imvigor_expr, metadata, FGM4_CYTOTOXIC,
                             'FGM4 (cytotoxic, raw)',
                             ax_km=axes[0, 0], ax_box=axes[0, 1])

    # FGM4 normalized by immune CGM core
    # (requires running normalize_by_core on ImVigor210 first)
    # imv_norm = normalize_by_core(imvigor_cgm_df, imv_core_genes)
    # res2 = survival_analysis(imv_norm[1], metadata, FGM4_CYTOTOXIC,
    #                          'FGM4 / CGM core (normalized)',
    #                          ax_km=axes[1,0], ax_box=axes[1,1])

    plt.suptitle('Survival and Response — ImVigor210 (anti-PD-L1)', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f'FGM4 raw:        log-rank p={res1["p_logrank"]:.3f}, AUC={res1["AUC"]:.2f}')
else:
    print('Skipping survival analysis — ImVigor210 data not found.')
    print('Key finding from the paper (Figure 10):')
    print('  FGM4 (cytotoxic) unnormalized:  low predictive power')
    print('  FGM4 / CGM core (normalized):   significant OS and response association')